# Modern Application Development – I: Comprehensive Lecture Notes  
**Professor Nitin Chandrachoodan, IIT Madras**  
**Week 7: Backend Systems, Data Search, Database Internals, Scaling & Security**

---

## Table of Contents
1. [The Memory Hierarchy: From Registers to Cold Storage](#1-the-memory-hierarchy)
2. [Fundamentals of Data Search and Complexity](#2-data-search)
3. [Database Search: Indexes, B-Trees, and Hashing](#3-database-search)
4. [SQL vs NoSQL: A Panorama of Data Stores](#4-sql-vs-nosql)
5. [Scaling Databases: Replication, Scale‑Up vs Scale‑Out, and BASE](#5-scaling)
6. [Security of Databases and Web Applications](#6-security)

---

## 1. The Memory Hierarchy: From Registers to Cold Storage

### 1.1 Why App Developers Must Understand Hardware
In previous weeks, we focused on the logical structure of apps—MVC, RESTful APIs, routes, and models. Now we step closer to the physical reality. Every database, every variable, every user session lives on real hardware. The performance, cost, and reliability of an application are deeply influenced by *where and how* data is stored. As an app grows, the choice of storage technology—and how it is used—can make the difference between a snappy experience and an unusable one.

### 1.2 The Layers of Memory
A modern computer system organises storage in a **hierarchy**, trading off speed, capacity, and cost per bit. From fastest (and smallest) to slowest (and largest):

1. **Registers**  
   - Located directly inside the CPU core.  
   - Number of registers is tiny (typically a few dozen, each holding 32 or 64 bits).  
   - Access latency is on the order of **a single CPU cycle (≈1 nanosecond)**.  
   - Used for the most immediate operands and results of arithmetic/logical operations.  
   - Completely volatile; managed by the compiler and processor pipeline.

2. **SRAM Cache (L1, L2, L3)**  
   - Static Random Access Memory built with fast transistor flip‑flops.  
   - L1 cache is the smallest (tens of KB) and fastest (≈1–2 ns).  
   - L2 and L3 are progressively larger (up to a few MB per core or shared) and slightly slower.  
   - The cache automatically stores copies of recently‑used main‑memory locations.  
   - Because of **temporal and spatial locality**, the cache drastically reduces the average memory access time.  
   - For an app developer, the fact that *sequential memory access is much faster than random access* is a direct consequence of cache behaviour.

3. **DRAM (Main Memory)**  
   - Dynamic RAM, the “RAM” of a PC or server (typically 4–32 GB).  
   - Access latency is around **50–100 ns**, roughly 50–100× slower than L1 cache.  
   - Volatile: contents are lost if power fails.  
   - Much denser and cheaper than SRAM, but still expensive compared to disk.  
   - All in‑memory data structures (Python lists, dictionaries, session caches, the working set of a database) live here.

4. **SSD (Solid‑State Drive)**  
   - Based on NAND flash memory; no moving parts.  
   - Latency for a random read is typically **50–100 µs** (microseconds), about 1000× slower than DRAM.  
   - Throughput can be very high (hundreds of MB/s), especially for sequential operations.  
   - Non‑volatile: data persists after power loss.  
   - Capacities range from 128 GB to several TB.  
   - Many databases now use SSDs as the primary persistent store.

5. **HDD (Hard Disk Drive)**  
   - Magnetic spinning platters and a moving read/write head.  
   - Seek time (moving the head) and rotational latency make random access extremely slow: **5–15 ms** (milliseconds). That is 100,000× slower than DRAM.  
   - Sequential throughput is reasonable (100–200 MB/s).  
   - Very cheap per GB; capacities up to 10+ TB.  
   - Traditionally the backbone of database storage, but increasingly replaced by SSDs for performance‑critical workloads.

6. **Cold Storage (Archive)**  
   - Tape libraries, optical disks, or highly redundant cloud object stores (Amazon Glacier, Google Cloud Archive).  
   - Retrieval can take **minutes to hours**.  
   - Extremely low cost per TB; designed for backups and long‑term retention where access is rare.  
   - Durability is exceptionally high (11 9s or more) through replication and error‑correcting codes.

### 1.3 The Cost–Performance Trade‑Off
The hierarchy exists because no single technology is simultaneously fast, cheap, and large. As an app developer, you indirectly control the hierarchy through your architectural decisions:
- Keep the *working set* (frequently accessed data) in DRAM—e.g., by using an in‑memory cache like Redis.
- Choose a database that stores its primary data on SSD for low latency.
- Archive old logs to cold storage to save costs.
- Understand that a single disk seek (HDD) can waste millions of CPU cycles—hence the importance of indexing and query optimisation.

---

## 2. Fundamentals of Data Search and Complexity

### 2.1 Big O Notation – A Quick Primer
The **Big O notation** (e.g., \(O(N)\), \(O(\log N)\)) describes how the time (or space) an algorithm requires grows as the size of the input \(N\) increases. It abstracts away constant factors and lower‑order terms to focus on the *asymptotic* behaviour.

| Notation | Name | Example |
|----------|------|---------|
| \(O(1)\) | Constant | Accessing an array element by index |
| \(O(\log N)\) | Logarithmic | Binary search |
| \(O(N)\) | Linear | Searching an unsorted list |
| \(O(N \log N)\) | Linearithmic | Efficient sorting (mergesort) |
| \(O(N^2)\) | Quadratic | Nested loops over all pairs |
| \(O(2^N)\) | Exponential | Brute‑force travelling salesman |

In the context of databases, we are typically interested in the complexity of **lookups** (finding a row by key), **range queries**, and **insertions/deletions**.

### 2.2 Searching in a Linked List
A linked list stores elements where each node points to the next. To find an element, we must start at the head and follow pointers until we find the target or reach the end. In the worst case, we visit all \(N\) nodes. Hence, **search is \(O(N)\)**. Linked lists are simple but unsuitable for fast lookups in large datasets.

### 2.3 Searching in an Unsorted Array
An array in memory provides **random access**: we can reach the \(i\)‑th element in constant time using the base address + offset. However, if we do not know *which* index holds the desired value, we must scan element by element. Again, **search is \(O(N)\)**. The constant factor is smaller than a linked list because of memory locality (cache benefits), but the asymptotic behaviour is still linear.

### 2.4 Binary Search in a Sorted Array
If the array is **sorted** (e.g., alphabetically or numerically), we can exploit the ordering. Binary search:
- Look at the middle element. If it is the target, stop.
- If the target is smaller, discard the right half; if larger, discard the left half.
- Repeat on the remaining half.

Each step halves the search space. After \(k\) steps, the remaining size is \(N / 2^k\). We stop when the size is 1, so \(2^k = N\), i.e., \(k = \log_2 N\). **Binary search is \(O(\log N)\)**.  
For \(N = 1,000,000\), \(\log_2 N \approx 20\) — a huge improvement over scanning a million entries.

### 2.5 Binary Search Trees (BST)
A BST stores data in nodes such that for any node:
- All values in the left subtree are **less** than the node’s value.
- All values in the right subtree are **greater**.

Searching in a BST follows the same logic as binary search: at each node, compare and go left or right. In a **balanced** BST, the height is \(O(\log N)\), so search is \(O(\log N)\). However, if data is inserted in sorted order, the tree degenerates into a linked list (height \(O(N)\)), making search \(O(N)\). **Self‑balancing trees** (AVL, Red‑Black) guarantee \(O(\log N)\) height by performing rotations on insert/delete.

### 2.6 B-Trees – The Disk‑Friendly Tree
B‑trees generalise BSTs. Instead of two children per node, a B‑tree node can have many children (e.g., 100). The node itself stores multiple keys in sorted order. This structure is designed for **block‑oriented storage** (like HDDs and SSDs):
- A single node typically matches the size of a disk block (e.g., 4 KB or 16 KB).
- By reading one block from disk, we can examine many keys and decide which child to follow next.
- The tree remains perfectly balanced (all leaves at the same depth).  
- Height is \(O(\log_B N)\) where \(B\) is the branching factor. For typical databases, a B‑tree of height 3 can index millions of rows.

B‑trees (and their variants, B+ trees) are the **most common index structure in relational databases** (MySQL, PostgreSQL, SQLite). They support point queries, range scans, and prefix searches efficiently.

### 2.7 Hash Tables
A hash table uses a **hash function** to compute an integer (the hash code) from the search key. That integer maps directly to a slot (bucket) in an array. If the hash function is good and the table is large enough, collisions are rare, and **search, insert, and delete are \(O(1)\) on average**.
- **Strengths:** Extremely fast for exact‑match lookups (“find user with id=42”).
- **Weaknesses:** Cannot perform range queries (“find all users with id between 100 and 200”), prefix searches, or ordered traversals. Hash tables also do not inherently keep data sorted.

In databases, hash indexes are available (e.g., in MySQL’s Memory engine) and are ideal for workloads dominated by equality comparisons.

---

## 3. Database Search: Indexes, B-Trees, and Hashing

### 3.1 The Need for Indexes
A relational database stores data in tables. Without any index, a `SELECT … WHERE column = value` must perform a **full table scan** (\(O(N)\)). For large tables, this is catastrophic for performance.

An **index** is a separate data structure (often a B‑tree or hash table) that stores a copy of one or more columns in sorted order, along with pointers (row IDs) back to the actual table rows. An index on column `X` allows the database to:
- Locate rows with a specific `X` in \(O(\log N)\) time (B‑tree) or \(O(1)\) (hash).
- Efficiently retrieve a range of values for `X` (B‑tree only).
- Avoid reading the entire table, drastically reducing I/O.

**Indexes come at a cost:** they consume additional disk space and slow down `INSERT`, `UPDATE`, and `DELETE` operations because the index must be updated alongside the table. Thus, choosing which columns to index is a critical performance optimisation.

### 3.2 How B-Tree Indexes Work for Different Queries
Consider a table `users` with a B‑tree index on `last_name`.

- **Equality:** `WHERE last_name = 'Smith'`  
  The B‑tree is traversed like a binary search: start at the root, compare, descend. Locates the first row with ‘Smith’ in \(O(\log N)\). If multiple rows match, they are clustered together in the leaf pages because the index is sorted.

- **Range:** `WHERE last_name >= 'M' AND last_name < 'N'`  
  Traverse to the first key `>= 'M'`, then scan forward along the leaf‑level linked list until the key exceeds `'N'`. Highly efficient.

- **Prefix (LIKE):** `WHERE last_name LIKE 'Pat%'`  
  A B‑tree can use the index because the prefix ordering is consistent with the sorted order. It can narrow down to the subtree starting with ‘Pat’ and scan from there.

- **Suffix (LIKE):** `WHERE last_name LIKE '%trick'`  
  The leading `%` means the value could start with anything. The index ordering is based on the *start* of the string, so the database cannot use the index. A full table scan is required.

- **OR conditions:** `WHERE last_name = 'Smith' OR first_name = 'John'`  
  Unless there are separate indexes on both columns and the database can combine them (bitmap index merge), an `OR` often prevents efficient index usage. Each condition may be fast individually, but combining disjoint sets is expensive. An `AND` is much more index‑friendly.

### 3.3 Multi-Column (Composite) Indexes
A composite index on `(col1, col2, col3)` creates a single B‑tree where the keys are sorted first by `col1`, then by `col2`, then by `col3`. It is like a phone book sorted by last name, then first name.

**Effective queries that use the composite index:**
- `WHERE col1 = ?` – uses the index fully on the leading column.
- `WHERE col1 = ? AND col2 = ?` – even better; narrows down to the exact segment of the tree.
- `WHERE col1 = ? AND col2 > ?` – can range‑scan the portion of the index.
- `WHERE col1 = ? AND col3 = ?` – can use `col1` part, but `col3` is *not contiguous* in the index because it is not preceded by `col2`. The database may still use the index for the first column and then filter the rest. This is less efficient than if `col2` were also constrained.

**Ineffective queries:**
- `WHERE col2 = ?` – skips the leading column; the index ordering does not help (unless an index skip‑scan is possible, but often a full table scan is better).
- `WHERE col1 = ? OR col2 = ?` – again, `OR` breaks the ability to use a single index efficiently.

**The rule of thumb:** For a composite index `(A, B, C)`, it can be used for queries that filter on `A`, `A + B`, or `A + B + C`. The columns must be used in order from the leftmost, and there should be no `OR` across different columns.

### 3.4 Hash Indexes in Databases
Some databases (e.g., MySQL with MEMORY engine, PostgreSQL with hash indexes) support hash indexes.
- **Equality:** `WHERE column = 42` – \(O(1)\), extremely fast.
- **Range/prefix/sort:** Not supported. The index is useless for `>`, `<`, `BETWEEN`, `LIKE`, `ORDER BY`.

Hash indexes are niche but powerful for workloads like session tables where lookups are always by exact session ID.

### 3.5 Query Optimisation and the Database Engine
Modern databases have sophisticated **query optimisers**. Given a SQL query, the optimiser:
- Considers multiple execution plans (e.g., which index to use, join order).
- Estimates the cost (CPU, I/O) of each plan using statistics about table sizes and value distribution (histograms).
- Chooses the plan with the lowest estimated cost.

As an app developer using an ORM like SQLAlchemy, you do not write the final SQL directly, but you *shape it* through the structure of your ORM queries. Understanding indexes helps you structure your application’s data access patterns so that the ORM generates optimisable SQL. For example, always providing a filter on the leading column of a composite index when possible, or avoiding `LIKE '%...'` in large tables.

---

## 4. SQL vs NoSQL: A Panorama of Data Stores

### 4.1 The Relational Model (SQL) and Its Limitations
Relational databases (RDBMS) store data in tables with a strict schema. Every row has the same columns. This works wonderfully when the data is uniform and relationships are well‑defined. However, it becomes awkward when:
- Entities have highly variable attributes (e.g., students who are hostellers vs. day scholars require different fields). Using nullable columns leads to sparse, wasteful tables.
- The schema changes frequently, requiring costly `ALTER TABLE` operations.
- The data is naturally hierarchical or graph‑like, requiring many JOINs.

### 4.2 Document Databases
Document databases (e.g., **MongoDB**, Amazon DocumentDB, CouchDB) store data as **documents** – typically JSON objects. A collection is a group of documents; there is no fixed schema. Each document can have its own set of fields.

**Example:** Two student documents:
```json
{ "student_id": 1, "name": "Alice", "hostel": {"name": "Ganga", "room": "A-101"} }
{ "student_id": 2, "name": "Bob", "day_scholar": {"vehicle_reg": "KA-01-1234"} }
```
- No null columns; only relevant information is stored.
- Documents can embed related data (e.g., a blog post and its comments as nested objects), reducing the need for JOINs.
- Queries can index on any field, even nested ones (`hostel.name`).
- **Trade‑offs:** They may sacrifice some ACID guarantees (though MongoDB has added multi‑document transactions). Joining across collections is less natural than in SQL; often data is denormalised to avoid joins.

### 4.3 Key‑Value Stores
Key‑value stores are the simplest data model: a giant dictionary mapping keys to values. The value is often opaque to the database (a blob). Examples: **Redis**, **memcached**, Amazon DynamoDB (in key‑value mode), BerkeleyDB.

- **Strengths:** Extremely fast for point lookups. Usually, the entire dataset resides in memory (Redis). Ideal for caching, session stores, shopping cart data, rate limiting.
- **Weaknesses:** No query language for filtering by value content. No relationships. Values must be retrieved by key, then parsed by the application.
- Often used as a **complement** to a relational database: the RDBMS is the source of truth; Redis caches frequently‑read data, reducing load on the primary database.

### 4.4 Columnar (Wide‑Column) Stores
Traditional RDBMS are *row‑oriented*: all columns of a row are stored together. This is optimal for transactional workloads (fetch/update a whole row). **Columnar databases** store each column separately. This is optimal for analytical queries that aggregate a few columns across millions of rows (e.g., “average marks per course”).

- **Column‑oriented OLAP:** Apache Cassandra, HBase, Google Bigtable, Amazon Redshift (columnar storage inside). Cassandra is a *wide‑column* store: rows can have thousands of columns, and the set of columns can differ per row. Data is partitioned across nodes.
- **Advantages:** Compression is highly effective because values in a column are often similar. Aggregate functions (SUM, AVG) scan only the necessary columns.

### 4.5 Graph Databases
Graph databases (e.g., **Neo4j**, Amazon Neptune, JanusGraph) model data as **nodes** (entities) and **edges** (relationships), both with properties. They excel at queries like:
- “Find all friends‑of‑friends of Alice”
- “Shortest path between two users”
- “Products frequently bought together”

In a relational database, such queries involve recursive JOINs that are hard to write and slow. Graph databases use specialised index‑free adjacency: each node physically stores pointers to its neighbours, making traversal extremely fast (\(O(1)\) per hop). They are the backbone of social networks, recommendation engines, and fraud detection.

### 4.6 Time‑Series Databases
Time‑series databases (TSDB) — e.g., **InfluxDB**, TimescaleDB (on PostgreSQL), Prometheus — are optimised for data that arrives as a stream of timestamped measurements (CPU load, stock prices, sensor readings).

- **Write‑heavy:** Ingest millions of data points per second.
- **Time‑based queries:** “Average temperature over the last hour”, “Max requests per minute over the last 30 days”. They have built‑in functions for downsampling, retention policies (automatically delete/aggregate old data), and continuous queries.
- Data is typically stored in time‑ordered shards, enabling efficient range scans and compression.

### 4.7 What Does “NoSQL” Really Mean?
The term **NoSQL** originally meant “No SQL” (as in, we reject the relational model). Over time, many NoSQL databases added SQL‑like query languages. Today, **NoSQL is better understood as “Not Only SQL”**. It denotes a class of databases that deviate from the classic relational/table model and often sacrifice some ACID properties (especially consistency) for greater scalability, flexibility, or performance.

### 4.8 ACID vs BASE
This is a fundamental philosophical split.

- **ACID** (Atomicity, Consistency, Isolation, Durability) — the gold standard of traditional RDBMS. Every transaction either completely succeeds or completely fails; the database moves from one consistent state to another; concurrent transactions are isolated; committed data survives crashes. ACID is essential for financial transactions.

- **BASE** (Basically Available, Soft state, Eventual consistency) — often associated with NoSQL.  
  - **Basically Available:** The system guarantees availability (responds to every request) even in the face of partial failures, perhaps at the cost of consistency.  
  - **Soft state:** The state of the system may change over time, even without input, because of eventual consistency.  
  - **Eventual consistency:** If no new updates are made, eventually all replicas will converge to the same state. But there is a window where different clients may see different data.

**Why choose BASE over ACID?**  
The famous **CAP theorem** states that a distributed system can simultaneously provide only two of three guarantees: **C**onsistency (all nodes see the same data at the same time), **A**vailability (every request receives a response), and **P**artition tolerance (the system works despite network partitions). Since partitions are inevitable in large networks, one must often choose between consistency and availability. Many NoSQL systems choose availability and partition tolerance, embracing eventual consistency.

**Real‑world example:** A Facebook user’s friend list. It is acceptable if, for a few seconds, two users see a slightly different state (eventual consistency). However, a wire transfer between bank accounts absolutely requires strong consistency.

---

## 5. Scaling Databases: Replication, Scale‑Up vs Scale‑Out, and BASE

### 5.1 Data Replication
**Replication** means maintaining multiple copies of the same data on different servers. Reasons include:
- **High availability:** If one server fails, another can take over.
- **Read scalability:** Read queries can be distributed across replicas, increasing throughput.
- **Geographic distribution:** Placing replicas closer to users reduces latency.

Replication can be synchronous or asynchronous. Synchronous replication guarantees strong consistency (all replicas are updated before a transaction completes) but adds latency. Asynchronous replication is faster but leads to eventual consistency.

### 5.2 Scale‑Up (Vertical Scaling)
Add more resources to a single server: faster CPU, more RAM, faster disks. Traditional RDBMS like Oracle and IBM Db2 are designed to excel at vertical scaling. They can efficiently use massive hardware (100s of CPU cores, TBs of RAM).
- **Pros:** Simpler application logic; no distributed complexity; ACID transactions across the entire dataset.
- **Cons:** Hardware becomes exponentially expensive; there’s a physical limit to how big a single machine can be; the server is a single point of failure.

### 5.3 Scale‑Out (Horizontal Scaling)
Distribute the data and load across many commodity servers (nodes). Add more nodes as needed. This is the approach championed by NoSQL databases and the cloud.
- **Pros:** Can scale almost indefinitely; uses cheap hardware; resilient to individual node failures; can elastically scale up/down in the cloud.
- **Cons:** Distributed systems are complex; maintaining ACID across nodes is very hard (the CAP theorem); developers must handle eventual consistency, sharding strategies, and distributed queries.

### 5.4 The Cloud and Scale‑Out
Cloud platforms (AWS, Google Cloud, Azure) make horizontal scaling practical. Services like Amazon RDS can create read replicas with a few clicks. Amazon DynamoDB (a NoSQL key‑value store) automatically partitions data across nodes and scales on demand. Google Cloud Datastore and Firebase do the same. The serverless model abstracts away the underlying infrastructure entirely.

This has profoundly influenced modern application architecture: **stateless application servers** (easily scaled horizontally) coupled with **managed database services** that handle replication and sharding. The app developer codes against an API (e.g., SQL endpoint, DynamoDB API) and the cloud provider worries about the physical scaling.

### 5.5 Consistency Models in Practice
- **Strong consistency:** After an update completes, all subsequent reads see the updated value. Typical in ACID RDBMS.
- **Eventual consistency:** If no new updates, all reads will eventually return the updated value. Used by DynamoDB (optionally), Cassandra, and many DNS systems.
- **Read‑your‑writes consistency:** A user always sees their own updates immediately, even if other users might not. Common in social media.

Choosing the right consistency model depends on the user experience: for a banking dashboard, strong consistency; for a twitter feed, eventual is acceptable.

---

## 6. Security of Databases and Web Applications

### 6.1 The Danger of SQL Injection
SQL injection is one of the oldest and most devastating web vulnerabilities. It occurs when user‑supplied input is concatenated directly into SQL queries without sanitisation.

**Example vulnerable Python code:**
```python
name = request.form['name']
password = request.form['password']
query = f"SELECT * FROM users WHERE name='{name}' AND password='{password}'"
```
If an attacker enters in the name field: `' OR '1'='1`  
The query becomes:
```sql
SELECT * FROM users WHERE name='' OR '1'='1' AND password=''
```
Since `'1'='1'` is always true, the condition effectively ignores both name and password, returning all users—potentially allowing login bypass.

Worse, an attacker can enter: `'; DROP TABLE users; --`  
This terminates the original query and executes a destructive command.

### 6.2 Preventing SQL Injection
The fundamental rule is: **never concatenate user input into SQL**. Use **parameterised queries** (prepared statements) instead.
```python
cursor.execute("SELECT * FROM users WHERE name=? AND password=?", (name, password))
```
The database driver treats the parameters as pure data, not as executable SQL. No amount of malicious characters can alter the query structure.

Frameworks and ORMs (SQLAlchemy, Django ORM) use parameterised queries by default. When you write `User.query.filter_by(username=name)`, the ORM safely binds the variable. This is a major reason to use established frameworks instead of raw SQL string building.

### 6.3 Input Validation Beyond SQL
Even with parameterised queries, validation is crucial:
- **Data type checks:** Ensure a numeric field really contains a number.
- **Format checks:** Email addresses, dates, phone numbers.
- **Length limits:** Avoid buffer overflows and excessively long inputs.
- **Character whitelisting:** Allow only expected character sets; reject control characters or non‑UTF‑8 sequences that could cause crashes.

Validation must happen **on the server side**, never rely solely on client‑side JavaScript, because an attacker can bypass the browser entirely and send arbitrary HTTP requests using tools like curl.

### 6.4 HTTPS: Securing the Channel
**HTTPS = HTTP over TLS (Transport Layer Security).** It provides three essential guarantees:
1. **Encryption:** All data between client and server is encrypted. Eavesdroppers cannot read passwords, cookies, or content.
2. **Integrity:** Data cannot be modified in transit without detection.
3. **Authentication:** The server presents a certificate that proves its identity (e.g., that you are really connected to `google.com` and not an imposter).

**Limitations of HTTPS:** It only protects the *pipe*. It does **not**:
- Prevent SQL injection or other application‑layer attacks.
- Validate that the content being sent is benign.
- Protect data after it arrives at the server (e.g., if an attacker gains access to the server itself).

Moreover, HTTPS makes caching by intermediate proxies harder, because the proxy cannot read the encrypted content to cache it. This is a performance trade‑off, but today, the security benefits far outweigh the drawbacks.

**Server certificates** are issued by Certificate Authorities (CAs) that browsers trust. The certificate binds a public key to a domain. The chain of trust ensures that a valid certificate for `iitm.ac.in` can only be obtained by someone who controls that domain.

### 6.5 Web Application Security – A Multi‑Layered Concern
Security is not a single feature; it is a property of every layer of the stack:
- **Application code:** SQL injection, cross‑site scripting (XSS), cross‑site request forgery (CSRF), insecure direct object references.
- **Framework/library:** Using patched, well‑maintained frameworks prevents many vulnerabilities.
- **Server configuration:** Operating system, web server (Nginx, Apache) must be hardened.
- **Database:** Access control, encryption at rest, regular backups.
- **Network:** Firewalls, DDoS protection, HTTPS, VPNs.

### 6.6 Summary of Best Practices for the App Developer
1. **Never trust user input.** Treat all HTTP request data as potentially malicious.
2. **Use parameterised queries / ORMs** for all database interactions.
3. **Validate all inputs on the server side:** type, format, length, allowed characters.
4. **Deploy HTTPS everywhere.**
5. **Keep dependencies updated** (Flask, SQLAlchemy, Python itself) to benefit from security patches.
6. **Understand the environment:** whether you run on your own server or a managed cloud platform (App Engine, Heroku), know who handles what security responsibility (shared responsibility model).
7. **Be aware of common attacks** (OWASP Top 10) and how your framework helps mitigate them.

---

## Conclusion of Week 7

This week deepened the understanding of what happens “under the hood” of the model layer in a web application. The journey from the physical memory hierarchy to abstract data structures clarified *why* indexes are critical and *how* different queries exploit or ignore them. The exploration of NoSQL revealed that “database” is no longer synonymous with “relational tables”; a rich ecosystem of specialised data stores exists, each trading off consistency, scalability, and flexibility in its own way. The concepts of scaling—vertical vs. horizontal, replication, eventual consistency—are no longer just theoretical; they are everyday decisions in modern cloud‑native development. Finally, the stark reality of security threats like SQL injection grounded the discussion: all the elegant architecture in the world is worthless if a single unsanitised input can destroy the database. 

Armed with this knowledge, an app developer can make informed choices about storage engines, index design, scaling strategies, and defensive coding—moving from merely “making it work” to building robust, performant, and secure applications.